# Expérience 1

Cette expérience MLflow teste une version plus robuste du dataset historique en colonnes. L'objectif est simple : comparer `RandomForest`, `XGBoost` et `XGBoost Random Forest` après avoir corrigé les principales sources de surapprentissage identifiées dans les itérations précédentes.


## Choix retenus contre l'overfitting

La configuration retenue pour cette expérience applique trois garde-fous :
- `area` n'est plus utilisée comme variable explicative ; elle sert uniquement à construire le split groupé ;
- l'historique de rendement est limité aux `3` années les plus récentes avant la cible ;
- l'évaluation finale se fait avec un `GroupShuffleSplit` par `area`, pour tester la généralisation sur des pays absents du train.

Cette révision ajoute maintenant deux briques supplémentaires pour confirmer ou nuancer le diagnostic d'overfitting :
- une **cross-validation groupée** sur le train, afin de mesurer la stabilité des performances hors échantillon avant le test final ;
- des **variantes plus régularisées** de `RandomForest`, `XGBoost` et `XGBoost Random Forest`, afin de vérifier si une baisse contrôlée de complexité réduit l'écart train / validation / test.


In [1]:
from datetime import date
from pathlib import Path
import json
import sqlite3
import shutil

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRFRegressor, XGBRegressor

from scripts.mlflow_logging import log_named_sklearn_model
from scripts.project_config import DEFAULT_CONFIG_PATH, load_preparation_config

SEED = 42
pd.set_option('display.max_columns', None)


/Users/steph/Code/Python/Jupyter/OCR_Projet12/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Le notebook relit la configuration centralisée du projet, produit un dataset historique large et journalise les résultats dans la base MLflow locale du projet.


In [2]:
config = load_preparation_config(ensure_dirs=True)
MIN_YEAR = int(config['MIN_YEAR'])
CURRENT_YEAR = date.today().year
YEARS = list(range(MIN_YEAR, CURRENT_YEAR + 1))
CV_N_SPLITS = 4

YIELD_PATH = config['YIELD_PATH']
RAINFALL_PATH = config['RAINFALL_PATH']
PESTICIDES_PATH = config['PESTICIDES_PATH']
TEMP_PATH = config['TEMP_PATH']
ARTIFACTS_DIR = config['ARTIFACTS_DIR']

EXPERIENCE_DIR = ARTIFACTS_DIR / 'experiments' / 'experience_1'
EXPERIENCE_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR = EXPERIENCE_DIR / 'cv'
CV_DIR.mkdir(parents=True, exist_ok=True)
EXPERIENCE_DATASET_PATH = EXPERIENCE_DIR / 'dataset_consolide_historique_colonnes.csv'
SOURCE_OVERVIEW_PATH = EXPERIENCE_DIR / 'source_overview.csv'
SOURCE_QUALITY_PATH = EXPERIENCE_DIR / 'source_quality.csv'
EXPERIENCE_SUMMARY_PATH = EXPERIENCE_DIR / 'experience_1_summary.csv'
MISSING_SUMMARY_PATH = EXPERIENCE_DIR / 'experience_1_missing_summary.csv'
MODEL_RESULTS_PATH = EXPERIENCE_DIR / 'model_results.csv'
FAMILY_RESULTS_PATH = EXPERIENCE_DIR / 'family_best_results.csv'
SEARCH_SPACE_PATH = EXPERIENCE_DIR / 'targeted_search_space.json'

MLFLOW_DB_PATH = ARTIFACTS_DIR / 'mlflow.db'
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.resolve()}"
MLFLOW_EXPERIMENT_NAME = 'experience_1'
MLFLOW_ARTIFACTS_DIR = ARTIFACTS_DIR / 'mlruns'
MLFLOW_EXPERIMENT_ARTIFACT_DIR = MLFLOW_ARTIFACTS_DIR / MLFLOW_EXPERIMENT_NAME
MLFLOW_EXPERIMENT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Configuration chargée depuis : {DEFAULT_CONFIG_PATH.resolve()}')
print(f'Fenêtre historique : {MIN_YEAR}-{CURRENT_YEAR}')
print(f'Nombre de folds CV groupée : {CV_N_SPLITS}')
print(f'Dossier expérience : {EXPERIENCE_DIR.resolve()}')
print(f'Tracking MLflow : {MLFLOW_TRACKING_URI}')


Configuration chargée depuis : /Users/steph/Code/Python/Jupyter/OCR_Projet12/config/project_paths.yaml
Fenêtre historique : 1990-2026
Nombre de folds CV groupée : 4
Dossier expérience : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_1
Tracking MLflow : sqlite:////Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/mlflow.db


## Chargement et harmonisation des sources annuelles

On repart des quatre fichiers annuels utilisés pour la consolidation. Les clés sont harmonisées en `area`, `crop`, `year`, puis les doublons annuels sont traités avant la projection en colonnes.


In [3]:
yield_source = pd.read_csv(YIELD_PATH)
rainfall_source = pd.read_csv(RAINFALL_PATH, na_values=['..'])
pesticides_source = pd.read_csv(PESTICIDES_PATH)
temp_source = pd.read_csv(TEMP_PATH)

source_overview = pd.DataFrame(
    [
        {'fichier': 'yield.csv', 'lignes': yield_source.shape[0], 'colonnes': yield_source.shape[1], 'nan_detectes': int(yield_source.isna().sum().sum())},
        {'fichier': 'rainfall.csv', 'lignes': rainfall_source.shape[0], 'colonnes': rainfall_source.shape[1], 'nan_detectes': int(rainfall_source.isna().sum().sum())},
        {'fichier': 'pesticides.csv', 'lignes': pesticides_source.shape[0], 'colonnes': pesticides_source.shape[1], 'nan_detectes': int(pesticides_source.isna().sum().sum())},
        {'fichier': 'temp.csv', 'lignes': temp_source.shape[0], 'colonnes': temp_source.shape[1], 'nan_detectes': int(temp_source.isna().sum().sum())},
    ]
)
source_overview.to_csv(SOURCE_OVERVIEW_PATH, index=False)

yield_clean = (
    yield_source.loc[:, ['Area', 'Item', 'Year', 'Value']]
    .rename(columns={'Area': 'area', 'Item': 'crop', 'Year': 'year', 'Value': 'target_yield_t_ha'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        crop=lambda df: df['crop'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        target_yield_t_ha=lambda df: pd.to_numeric(df['target_yield_t_ha'], errors='coerce') / 10000,
    )
    .dropna(subset=['area', 'crop', 'year'])
)
yield_clean = yield_clean.loc[yield_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
yield_clean['year'] = yield_clean['year'].astype(int)
TARGET_YEAR = int(yield_clean['year'].max())
FEATURE_YEARS = [year for year in YEARS if year < TARGET_YEAR]
SELECTED_YIELD_YEARS = FEATURE_YEARS[-3:]

rainfall_clean = (
    rainfall_source.loc[:, [' Area', 'Year', 'average_rain_fall_mm_per_year']]
    .rename(columns={' Area': 'area', 'Year': 'year'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        average_rain_fall_mm_per_year=lambda df: pd.to_numeric(df['average_rain_fall_mm_per_year'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
rainfall_clean = rainfall_clean.loc[rainfall_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
rainfall_clean['year'] = rainfall_clean['year'].astype(int)
rainfall_clean = rainfall_clean.drop_duplicates(subset=['area', 'year'], keep='first')

pesticides_clean = (
    pesticides_source.loc[:, ['Area', 'Year', 'Value']]
    .rename(columns={'Area': 'area', 'Year': 'year', 'Value': 'pesticides_tonnes'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        pesticides_tonnes=lambda df: pd.to_numeric(df['pesticides_tonnes'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
pesticides_clean = pesticides_clean.loc[pesticides_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
pesticides_clean['year'] = pesticides_clean['year'].astype(int)
pesticides_clean = pesticides_clean.drop_duplicates(subset=['area', 'year'], keep='first')

temp_clean = (
    temp_source.loc[:, ['year', 'country', 'avg_temp']]
    .rename(columns={'country': 'area'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        avg_temp=lambda df: pd.to_numeric(df['avg_temp'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
temp_clean = temp_clean.loc[temp_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
temp_clean['year'] = temp_clean['year'].astype(int)
temp_clean = temp_clean.groupby(['area', 'year'], as_index=False)['avg_temp'].mean()

source_quality = pd.DataFrame(
    [
        {'table': 'yield_clean', 'clé': 'area + crop + year', 'doublons_sur_cle': int(yield_clean.duplicated(subset=['area', 'crop', 'year']).sum()), 'nan_totaux': int(yield_clean.isna().sum().sum())},
        {'table': 'rainfall_clean', 'clé': 'area + year', 'doublons_sur_cle': int(rainfall_clean.duplicated(subset=['area', 'year']).sum()), 'nan_totaux': int(rainfall_clean.isna().sum().sum())},
        {'table': 'pesticides_clean', 'clé': 'area + year', 'doublons_sur_cle': int(pesticides_clean.duplicated(subset=['area', 'year']).sum()), 'nan_totaux': int(pesticides_clean.isna().sum().sum())},
        {'table': 'temp_clean', 'clé': 'area + year', 'doublons_sur_cle': int(temp_clean.duplicated(subset=['area', 'year']).sum()), 'nan_totaux': int(temp_clean.isna().sum().sum())},
    ]
)
source_quality.to_csv(SOURCE_QUALITY_PATH, index=False)

display(source_overview)
display(source_quality)
print(f'Année cible retenue : {TARGET_YEAR}')
print(f'Années de rendement retenues comme features : {SELECTED_YIELD_YEARS}')


,fichier,lignes,colonnes,nan_detectes
0,yield.csv,56717,12,0
1,rainfall.csv,6727,3,780
2,pesticides.csv,4349,7,0
3,temp.csv,71311,3,2547


,table,clé,doublons_sur_cle,nan_totaux
0,yield_clean,area + crop + year,0,0
1,rainfall_clean,area + year,0,677
2,pesticides_clean,area + year,0,0
3,temp_clean,area + year,0,0


Année cible retenue : 2016
Années de rendement retenues comme features : [2013, 2014, 2015]


## Dataset historique en colonnes

Chaque variable annuelle est projetée en colonnes. La base finale contient une ligne par couple `area + crop`. La cible de modélisation est le rendement de `2016`, prédit à partir des rendements `2013-2015` et de l'historique climatique disponible.


In [4]:
def pivot_history(df: pd.DataFrame, index_cols: list[str], value_col: str, years: list[int]) -> pd.DataFrame:
    wide = df.pivot_table(index=index_cols, columns='year', values=value_col, aggfunc='first')
    wide = wide.reindex(columns=years)
    wide.columns = [f'{value_col}_{int(year)}' for year in wide.columns]
    return wide.reset_index()

base_keys = (
    yield_clean[['area', 'crop']]
    .drop_duplicates()
    .sort_values(['area', 'crop'])
    .reset_index(drop=True)
)

yield_history_wide = base_keys.merge(
    pivot_history(yield_clean, ['area', 'crop'], 'target_yield_t_ha', YEARS),
    on=['area', 'crop'],
    how='left',
    validate='1:1',
)
rainfall_history_wide = pivot_history(rainfall_clean, ['area'], 'average_rain_fall_mm_per_year', YEARS)
pesticides_history_wide = pivot_history(pesticides_clean, ['area'], 'pesticides_tonnes', YEARS)
temp_history_wide = pivot_history(temp_clean, ['area'], 'avg_temp', YEARS)

experience_1_dataset = (
    yield_history_wide
    .merge(rainfall_history_wide, on='area', how='left', validate='m:1')
    .merge(pesticides_history_wide, on='area', how='left', validate='m:1')
    .merge(temp_history_wide, on='area', how='left', validate='m:1')
    .sort_values(['area', 'crop'])
    .reset_index(drop=True)
)

missing_summary = (
    experience_1_dataset.isna()
    .sum()
    .rename('nb_nan')
    .reset_index()
    .rename(columns={'index': 'variable'})
)
missing_summary['part_nan_pct'] = (missing_summary['nb_nan'] / len(experience_1_dataset) * 100).round(2)
missing_summary.to_csv(MISSING_SUMMARY_PATH, index=False)

experience_summary = pd.DataFrame(
    {
        'indicateur': [
            'nb_lignes',
            'nb_colonnes',
            'annee_cible_modele',
            'part_nan_globale_pct',
            'colonnes_cible_historiques',
            'colonnes_pluie_historiques',
            'colonnes_pesticides_historiques',
            'colonnes_temperature_historiques',
        ],
        'valeur': [
            int(experience_1_dataset.shape[0]),
            int(experience_1_dataset.shape[1]),
            TARGET_YEAR,
            round(experience_1_dataset.isna().mean().mean() * 100, 2),
            len([c for c in experience_1_dataset.columns if c.startswith('target_yield_t_ha_')]),
            len([c for c in experience_1_dataset.columns if c.startswith('average_rain_fall_mm_per_year_')]),
            len([c for c in experience_1_dataset.columns if c.startswith('pesticides_tonnes_')]),
            len([c for c in experience_1_dataset.columns if c.startswith('avg_temp_')]),
        ],
    }
)
experience_summary.to_csv(EXPERIENCE_SUMMARY_PATH, index=False)
experience_1_dataset.to_csv(EXPERIENCE_DATASET_PATH, index=False)

display(experience_summary)
display(missing_summary.head(15))
print(f'Dataset expérience 1 sauvegardé : {EXPERIENCE_DATASET_PATH.resolve()}')


,indicateur,valeur
0,nb_lignes,1168.00
1,nb_colonnes,150.00
2,annee_cible_modele,2016.00
3,part_nan_globale_pct,44.51
4,colonnes_cible_historiques,37.00
5,colonnes_pluie_historiques,37.00
6,colonnes_pesticides_historiques,37.00
7,colonnes_temperature_historiques,37.00


,variable,nb_nan,part_nan_pct
0,area,0,0.00
1,crop,0,0.00
2,target_yield_t_ha_1990,179,15.33
3,target_yield_t_ha_1991,181,15.50
4,target_yield_t_ha_1992,102,8.73
5,target_yield_t_ha_1993,93,7.96
6,target_yield_t_ha_1994,89,7.62
7,target_yield_t_ha_1995,83,7.11
8,target_yield_t_ha_1996,85,7.28
9,target_yield_t_ha_1997,81,6.93


Dataset expérience 1 sauvegardé : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_1/dataset_consolide_historique_colonnes.csv


## Jeu de modélisation retenu

Ici, `area` est retirée des features mais conservée pour le `GroupShuffleSplit`. Le one-hot encoding ne porte plus que sur `crop`, ce qui réduit fortement la taille de la matrice encodée. Les colonnes numériques entièrement vides côté train sont supprimées avant entraînement.


In [5]:
target_col = f'target_yield_t_ha_{TARGET_YEAR}'

full_feature_pool_cols = ['area', 'crop']
full_feature_pool_cols += [f'target_yield_t_ha_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols += [f'average_rain_fall_mm_per_year_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols += [f'pesticides_tonnes_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols += [f'avg_temp_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols = [col for col in full_feature_pool_cols if col in experience_1_dataset.columns]

feature_cols = ['crop']
feature_cols += [f'target_yield_t_ha_{year}' for year in SELECTED_YIELD_YEARS]
feature_cols += [f'average_rain_fall_mm_per_year_{year}' for year in FEATURE_YEARS]
feature_cols += [f'pesticides_tonnes_{year}' for year in FEATURE_YEARS]
feature_cols += [f'avg_temp_{year}' for year in FEATURE_YEARS]
feature_cols = [col for col in feature_cols if col in experience_1_dataset.columns]

base_columns = ['area'] + [col for col in feature_cols if col != 'area'] + [target_col]
base_columns = list(dict.fromkeys(base_columns))
model_df = experience_1_dataset[base_columns].copy()
model_df = model_df.dropna(subset=[target_col]).reset_index(drop=True)

categorical_features = ['crop']
numeric_features = [col for col in feature_cols if col not in categorical_features]

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()
groups = model_df['area'].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

train_empty_numeric_features = [col for col in numeric_features if not X_train[col].notna().any()]
numeric_features = [col for col in numeric_features if col not in train_empty_numeric_features]
feature_cols = categorical_features + numeric_features
X_train = X_train[feature_cols].copy()
X_test = X_test[feature_cols].copy()

shared_areas = sorted(set(groups_train) & set(groups_test))

modeling_overview = pd.DataFrame(
    {
        'indicateur': [
            'train_rows',
            'test_rows',
            'train_areas',
            'test_areas',
            'shared_areas_between_train_and_test',
            'selected_feature_count',
            'selected_yield_year_count',
            'numeric_feature_count',
            'categorical_feature_count',
            'area_used_as_feature',
            'area_used_as_group_only',
            'dropped_train_empty_numeric_features',
        ],
        'valeur': [
            X_train.shape[0],
            X_test.shape[0],
            groups_train.nunique(),
            groups_test.nunique(),
            len(shared_areas),
            len(feature_cols),
            len(SELECTED_YIELD_YEARS),
            len(numeric_features),
            len(categorical_features),
            0,
            1,
            len(train_empty_numeric_features),
        ],
    }
)
display(modeling_overview)
display(pd.DataFrame({'selected_yield_year': SELECTED_YIELD_YEARS}))
if train_empty_numeric_features:
    display(pd.DataFrame({'dropped_train_empty_numeric_feature': train_empty_numeric_features}))
print(f'Cible : {target_col}')


,indicateur,valeur
0,train_rows,903
1,test_rows,208
2,train_areas,163
3,test_areas,41
4,shared_areas_between_train_and_test,0
5,selected_feature_count,79
6,selected_yield_year_count,3
7,numeric_feature_count,78
8,categorical_feature_count,1
9,area_used_as_feature,0


,selected_yield_year
0,2013
1,2014
2,2015


,dropped_train_empty_numeric_feature
0,average_rain_fall_mm_per_year_2003
1,avg_temp_2014
2,avg_temp_2015


Cible : target_yield_t_ha_2016


## Encodage et dimension finale

Le `one-hot encoding` ne porte plus que sur `crop`. Cette réduction de cardinalité est un des choix centraux faits pour limiter la mémorisation du train.


In [6]:
def build_preprocessor() -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
        ],
        sparse_threshold=0.0,
    )

probe_preprocessor = build_preprocessor()
X_train_prepared = probe_preprocessor.fit_transform(X_train)
onehot_encoder = probe_preprocessor.named_transformers_['cat'].named_steps['onehot']
onehot_modalities = int(sum(len(categories) for categories in onehot_encoder.categories_))
encoded_feature_count = int(X_train_prepared.shape[1])

ohe_overview = pd.DataFrame(
    {
        'indicateur': ['raw_feature_count', 'onehot_modalities', 'encoded_feature_count'],
        'valeur': [len(feature_cols), onehot_modalities, encoded_feature_count],
    }
)
display(ohe_overview)


,indicateur,valeur
0,raw_feature_count,79
1,onehot_modalities,10
2,encoded_feature_count,88


## Entraînement, cross-validation, recherche ciblée et suivi MLflow

Les modèles sont comparés sur la même base et la même préparation.
Cette version ajoute un **diagnostic en deux étages** :

- une `GroupKFold` sur le train pour confirmer le niveau de stabilité hors échantillon ;
- un test final séparé sur des `area` non vues, comme dans la version précédente.

En plus des baselines déjà présentes, le notebook lance maintenant une **recherche ciblée et manuelle** sur les deux familles les plus prometteuses à ce stade :
- `RandomForest`, qui a déjà montré le meilleur compromis biais / variance ;
- `XGBoost Random Forest`, qui réduit bien l'overfitting mais au prix d'une perte de performance qu'on cherche ici à corriger.


In [7]:
def build_model_pipeline(estimator) -> Pipeline:
    return Pipeline(
        steps=[
            ('preprocessor', build_preprocessor()),
            ('regressor', clone(estimator)),
        ]
    )


def compute_regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
    }


def run_group_cross_validation(estimator, X_data, y_data, groups_data, n_splits: int = CV_N_SPLITS):
    unique_group_count = int(groups_data.nunique())
    effective_splits = min(n_splits, unique_group_count)
    if effective_splits < 2:
        raise ValueError('La cross-validation groupée nécessite au moins 2 groupes distincts.')

    cv = GroupKFold(n_splits=effective_splits)
    fold_rows = []

    for fold_id, (cv_train_idx, cv_val_idx) in enumerate(cv.split(X_data, y_data, groups=groups_data), start=1):
        X_cv_train = X_data.iloc[cv_train_idx].reset_index(drop=True)
        X_cv_val = X_data.iloc[cv_val_idx].reset_index(drop=True)
        y_cv_train = y_data.iloc[cv_train_idx].reset_index(drop=True)
        y_cv_val = y_data.iloc[cv_val_idx].reset_index(drop=True)
        groups_cv_train = groups_data.iloc[cv_train_idx].reset_index(drop=True)
        groups_cv_val = groups_data.iloc[cv_val_idx].reset_index(drop=True)

        pipeline = build_model_pipeline(estimator)
        pipeline.fit(X_cv_train, y_cv_train)

        train_pred = pipeline.predict(X_cv_train)
        val_pred = pipeline.predict(X_cv_val)

        train_metrics = compute_regression_metrics(y_cv_train, train_pred)
        val_metrics = compute_regression_metrics(y_cv_val, val_pred)
        fold_rows.append(
            {
                'fold': fold_id,
                'train_rows': int(len(X_cv_train)),
                'val_rows': int(len(X_cv_val)),
                'train_areas': int(groups_cv_train.nunique()),
                'val_areas': int(groups_cv_val.nunique()),
                'train_mae': train_metrics['mae'],
                'train_rmse': train_metrics['rmse'],
                'train_r2': train_metrics['r2'],
                'val_mae': val_metrics['mae'],
                'val_rmse': val_metrics['rmse'],
                'val_r2': val_metrics['r2'],
                'overfit_gap_rmse': float(val_metrics['rmse'] - train_metrics['rmse']),
                'overfit_ratio_rmse': float(val_metrics['rmse'] / train_metrics['rmse']) if train_metrics['rmse'] > 0 else np.nan,
            }
        )

    cv_fold_df = pd.DataFrame(fold_rows)
    cv_summary = {
        'cv_n_splits': int(effective_splits),
        'cv_train_mae_mean': float(cv_fold_df['train_mae'].mean()),
        'cv_train_rmse_mean': float(cv_fold_df['train_rmse'].mean()),
        'cv_train_r2_mean': float(cv_fold_df['train_r2'].mean()),
        'cv_val_mae_mean': float(cv_fold_df['val_mae'].mean()),
        'cv_val_mae_std': float(cv_fold_df['val_mae'].std(ddof=0)),
        'cv_val_rmse_mean': float(cv_fold_df['val_rmse'].mean()),
        'cv_val_rmse_std': float(cv_fold_df['val_rmse'].std(ddof=0)),
        'cv_val_r2_mean': float(cv_fold_df['val_r2'].mean()),
        'cv_val_r2_std': float(cv_fold_df['val_r2'].std(ddof=0)),
        'cv_overfit_gap_rmse_mean': float(cv_fold_df['overfit_gap_rmse'].mean()),
        'cv_overfit_ratio_rmse_mean': float(cv_fold_df['overfit_ratio_rmse'].mean()),
    }
    return cv_fold_df, cv_summary


def register_candidate(candidate_registry: dict, model_name: str, estimator, *, model_family: str, search_family: str, regularization_profile: str, tuning_stage: str) -> None:
    candidate_registry[model_name] = {
        'estimator': estimator,
        'model_family': model_family,
        'search_family': search_family,
        'regularization_profile': regularization_profile,
        'tuning_stage': tuning_stage,
    }


candidate_models: dict[str, dict[str, object]] = {}
search_space_definition = {
    'random_forest_focus': [
        {
            'model_name': 'random_forest',
            'tuning_stage': 'baseline',
            'params': {
                'n_estimators': 300,
                'max_depth': 12,
                'min_samples_leaf': 2,
                'min_samples_split': 2,
                'max_features': 1.0,
            },
        },
        {
            'model_name': 'random_forest_regularized',
            'tuning_stage': 'seed_regularized',
            'params': {
                'n_estimators': 250,
                'max_depth': 8,
                'min_samples_leaf': 4,
                'min_samples_split': 8,
                'max_features': 0.6,
            },
        },
        {
            'model_name': 'random_forest_search_01',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 350,
                'max_depth': 9,
                'min_samples_leaf': 4,
                'min_samples_split': 10,
                'max_features': 0.5,
            },
        },
        {
            'model_name': 'random_forest_search_02',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 300,
                'max_depth': 7,
                'min_samples_leaf': 6,
                'min_samples_split': 12,
                'max_features': 0.7,
            },
        },
        {
            'model_name': 'random_forest_search_03',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 420,
                'max_depth': 8,
                'min_samples_leaf': 5,
                'min_samples_split': 10,
                'max_features': 0.45,
            },
        },
    ],
    'xgboost_focus': [
        {
            'model_name': 'xgboost',
            'tuning_stage': 'baseline',
            'params': {
                'n_estimators': 300,
                'max_depth': 6,
                'learning_rate': 0.05,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'reg_lambda': 1.0,
                'min_child_weight': 1,
                'reg_alpha': 0.0,
                'gamma': 0.0,
            },
        },
        {
            'model_name': 'xgboost_regularized',
            'tuning_stage': 'seed_regularized',
            'params': {
                'n_estimators': 220,
                'max_depth': 4,
                'learning_rate': 0.04,
                'subsample': 0.75,
                'colsample_bytree': 0.75,
                'reg_lambda': 3.0,
                'min_child_weight': 6,
                'reg_alpha': 0.3,
                'gamma': 0.1,
            },
        },
    ],
    'xgboost_random_forest_focus': [
        {
            'model_name': 'xgboost_random_forest',
            'tuning_stage': 'baseline',
            'params': {
                'n_estimators': 400,
                'max_depth': 8,
                'learning_rate': 1.0,
                'subsample': 0.8,
                'colsample_bynode': 0.8,
                'reg_lambda': 1.0,
                'min_child_weight': 1,
                'reg_alpha': 0.0,
            },
        },
        {
            'model_name': 'xgboost_random_forest_regularized',
            'tuning_stage': 'seed_regularized',
            'params': {
                'n_estimators': 260,
                'max_depth': 5,
                'learning_rate': 1.0,
                'subsample': 0.7,
                'colsample_bynode': 0.7,
                'reg_lambda': 3.0,
                'min_child_weight': 6,
                'reg_alpha': 0.2,
            },
        },
        {
            'model_name': 'xgboost_random_forest_search_01',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 320,
                'max_depth': 6,
                'learning_rate': 1.0,
                'subsample': 0.75,
                'colsample_bynode': 0.75,
                'reg_lambda': 4.0,
                'min_child_weight': 8,
                'reg_alpha': 0.1,
            },
        },
        {
            'model_name': 'xgboost_random_forest_search_02',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 300,
                'max_depth': 4,
                'learning_rate': 1.0,
                'subsample': 0.65,
                'colsample_bynode': 0.65,
                'reg_lambda': 6.0,
                'min_child_weight': 10,
                'reg_alpha': 0.4,
            },
        },
        {
            'model_name': 'xgboost_random_forest_search_03',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 220,
                'max_depth': 5,
                'learning_rate': 1.0,
                'subsample': 0.8,
                'colsample_bynode': 0.6,
                'reg_lambda': 4.0,
                'min_child_weight': 8,
                'reg_alpha': 0.3,
            },
        },
        {
            'model_name': 'xgboost_random_forest_search_04',
            'tuning_stage': 'targeted_search',
            'params': {
                'n_estimators': 280,
                'max_depth': 5,
                'learning_rate': 1.0,
                'subsample': 0.72,
                'colsample_bynode': 0.68,
                'reg_lambda': 5.0,
                'min_child_weight': 7,
                'reg_alpha': 0.15,
            },
        },
    ],
}

for spec in search_space_definition['random_forest_focus']:
    params = spec['params']
    register_candidate(
        candidate_models,
        spec['model_name'],
        RandomForestRegressor(
            random_state=SEED,
            n_jobs=-1,
            **params,
        ),
        model_family='random_forest',
        search_family='random_forest_focus',
        regularization_profile='manual_targeted_search' if spec['tuning_stage'] == 'targeted_search' else spec['tuning_stage'],
        tuning_stage=spec['tuning_stage'],
    )

for spec in search_space_definition['xgboost_focus']:
    params = spec['params']
    register_candidate(
        candidate_models,
        spec['model_name'],
        XGBRegressor(
            objective='reg:squarederror',
            tree_method='hist',
            random_state=SEED,
            n_jobs=-1,
            **params,
        ),
        model_family='xgboost',
        search_family='xgboost_focus',
        regularization_profile='manual_targeted_search' if spec['tuning_stage'] == 'targeted_search' else spec['tuning_stage'],
        tuning_stage=spec['tuning_stage'],
    )

for spec in search_space_definition['xgboost_random_forest_focus']:
    params = spec['params']
    register_candidate(
        candidate_models,
        spec['model_name'],
        XGBRFRegressor(
            objective='reg:squarederror',
            tree_method='hist',
            random_state=SEED,
            n_jobs=-1,
            **params,
        ),
        model_family='xgboost_random_forest',
        search_family='xgboost_random_forest_focus',
        regularization_profile='manual_targeted_search' if spec['tuning_stage'] == 'targeted_search' else spec['tuning_stage'],
        tuning_stage=spec['tuning_stage'],
    )

SEARCH_SPACE_PATH.write_text(json.dumps(search_space_definition, indent=2), encoding='utf-8')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
while mlflow.active_run() is not None:
    mlflow.end_run()

tracking_db_path = Path(MLFLOW_TRACKING_URI.removeprefix('sqlite:///'))
experiment_artifact_uri = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve().as_uri()
tracking_db_path.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if tracking_db_path.exists():
    connection = sqlite3.connect(tracking_db_path)
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='experiments'")
    if cursor.fetchone() is not None:
        cursor.execute(
            'SELECT experiment_id, name, artifact_location FROM experiments WHERE name = ?',
            (MLFLOW_EXPERIMENT_NAME,),
        )
        existing_row = cursor.fetchone()
        if existing_row is not None:
            experiment_id, _, current_artifact_location = existing_row
            current_artifact_dir = Path(str(current_artifact_location).removeprefix('file://')).resolve()
            target_artifact_dir = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve()
            if current_artifact_dir.exists() and current_artifact_dir != target_artifact_dir:
                for child in current_artifact_dir.iterdir():
                    destination = target_artifact_dir / child.name
                    if not destination.exists():
                        shutil.move(str(child), str(destination))
                if current_artifact_dir.exists() and current_artifact_dir.is_dir() and not any(current_artifact_dir.iterdir()):
                    current_artifact_dir.rmdir()
            cursor.execute(
                'UPDATE experiments SET artifact_location = ? WHERE experiment_id = ?',
                (experiment_artifact_uri, experiment_id),
            )
            cursor.execute(
                'UPDATE runs SET artifact_uri = REPLACE(artifact_uri, ?, ?) WHERE experiment_id = ? AND artifact_uri LIKE ?',
                (
                    str(current_artifact_dir),
                    str(target_artifact_dir),
                    experiment_id,
                    f'{current_artifact_dir}%',
                ),
            )
            connection.commit()
    connection.close()

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    client.create_experiment(MLFLOW_EXPERIMENT_NAME, artifact_location=experiment_artifact_uri)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

results = []
for model_name, model_spec in candidate_models.items():
    estimator = model_spec['estimator']
    estimator_params = estimator.get_params()
    cv_fold_df, cv_summary = run_group_cross_validation(estimator, X_train, y_train, groups_train)

    pipeline = build_model_pipeline(estimator)
    with mlflow.start_run(run_name=f'experience_1__{model_name}') as run:
        pipeline.fit(X_train, y_train)

        train_pred = pipeline.predict(X_train)
        test_pred = pipeline.predict(X_test)

        train_metrics = compute_regression_metrics(y_train, train_pred)
        test_metrics = compute_regression_metrics(y_test, test_pred)
        overfit_gap_rmse = float(test_metrics['rmse'] - train_metrics['rmse'])
        overfit_ratio_rmse = float(test_metrics['rmse'] / train_metrics['rmse']) if train_metrics['rmse'] > 0 else np.nan

        cv_fold_path = CV_DIR / f'{model_name}_cv_folds.csv'
        cv_summary_path = CV_DIR / f'{model_name}_cv_summary.json'
        model_params_path = CV_DIR / f'{model_name}_params.json'
        cv_fold_df.to_csv(cv_fold_path, index=False)
        cv_summary_path.write_text(json.dumps(cv_summary, indent=2), encoding='utf-8')
        model_params_path.write_text(json.dumps(estimator_params, default=str, indent=2), encoding='utf-8')

        mlflow.log_param('experience_name', 'experience_1')
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('model_family', model_spec['model_family'])
        mlflow.log_param('search_family', model_spec['search_family'])
        mlflow.log_param('tuning_stage', model_spec['tuning_stage'])
        mlflow.log_param('target_col', target_col)
        mlflow.log_param('target_year', TARGET_YEAR)
        mlflow.log_param('split_strategy', 'GroupShuffleSplit(area)')
        mlflow.log_param('cross_validation_strategy', 'GroupKFold(area)_on_train_only')
        mlflow.log_param('area_used_as_feature', False)
        mlflow.log_param('selected_feature_count', len(feature_cols))
        mlflow.log_param('numeric_feature_count', len(numeric_features))
        mlflow.log_param('categorical_feature_count', len(categorical_features))
        mlflow.log_param('dropped_train_empty_numeric_feature_count', len(train_empty_numeric_features))
        mlflow.log_param('selected_yield_year_count', len(SELECTED_YIELD_YEARS))
        mlflow.log_param('selected_yield_year_start', int(SELECTED_YIELD_YEARS[0]))
        mlflow.log_param('selected_yield_year_end', int(SELECTED_YIELD_YEARS[-1]))
        mlflow.log_param('encoded_feature_count', encoded_feature_count)
        mlflow.log_param('onehot_modalities', onehot_modalities)
        mlflow.log_param('cv_n_splits', cv_summary['cv_n_splits'])
        mlflow.log_param('regularization_profile', model_spec['regularization_profile'])

        mlflow.log_metric('train_mae', train_metrics['mae'])
        mlflow.log_metric('train_rmse', train_metrics['rmse'])
        mlflow.log_metric('train_r2', train_metrics['r2'])
        mlflow.log_metric('test_mae', test_metrics['mae'])
        mlflow.log_metric('test_rmse', test_metrics['rmse'])
        mlflow.log_metric('test_r2', test_metrics['r2'])
        mlflow.log_metric('overfit_gap_rmse', overfit_gap_rmse)
        mlflow.log_metric('overfit_ratio_rmse', overfit_ratio_rmse)
        mlflow.log_metric('cv_train_mae_mean', cv_summary['cv_train_mae_mean'])
        mlflow.log_metric('cv_train_rmse_mean', cv_summary['cv_train_rmse_mean'])
        mlflow.log_metric('cv_train_r2_mean', cv_summary['cv_train_r2_mean'])
        mlflow.log_metric('cv_val_mae_mean', cv_summary['cv_val_mae_mean'])
        mlflow.log_metric('cv_val_mae_std', cv_summary['cv_val_mae_std'])
        mlflow.log_metric('cv_val_rmse_mean', cv_summary['cv_val_rmse_mean'])
        mlflow.log_metric('cv_val_rmse_std', cv_summary['cv_val_rmse_std'])
        mlflow.log_metric('cv_val_r2_mean', cv_summary['cv_val_r2_mean'])
        mlflow.log_metric('cv_val_r2_std', cv_summary['cv_val_r2_std'])
        mlflow.log_metric('cv_overfit_gap_rmse_mean', cv_summary['cv_overfit_gap_rmse_mean'])
        mlflow.log_metric('cv_overfit_ratio_rmse_mean', cv_summary['cv_overfit_ratio_rmse_mean'])

        mlflow.log_artifact(str(EXPERIENCE_SUMMARY_PATH))
        mlflow.log_artifact(str(MISSING_SUMMARY_PATH))
        mlflow.log_artifact(str(EXPERIENCE_DATASET_PATH))
        mlflow.log_artifact(str(SEARCH_SPACE_PATH))
        mlflow.log_artifact(str(cv_fold_path))
        mlflow.log_artifact(str(cv_summary_path))
        mlflow.log_artifact(str(model_params_path))
        log_named_sklearn_model(pipeline, model_name=model_name)

        results.append(
            {
                'model': model_name,
                'model_family': model_spec['model_family'],
                'search_family': model_spec['search_family'],
                'tuning_stage': model_spec['tuning_stage'],
                'regularization_profile': model_spec['regularization_profile'],
                'train_mae': train_metrics['mae'],
                'train_rmse': train_metrics['rmse'],
                'train_r2': train_metrics['r2'],
                'cv_val_mae_mean': cv_summary['cv_val_mae_mean'],
                'cv_val_mae_std': cv_summary['cv_val_mae_std'],
                'cv_val_rmse_mean': cv_summary['cv_val_rmse_mean'],
                'cv_val_rmse_std': cv_summary['cv_val_rmse_std'],
                'cv_val_r2_mean': cv_summary['cv_val_r2_mean'],
                'cv_overfit_gap_rmse_mean': cv_summary['cv_overfit_gap_rmse_mean'],
                'cv_overfit_ratio_rmse_mean': cv_summary['cv_overfit_ratio_rmse_mean'],
                'test_mae': test_metrics['mae'],
                'test_rmse': test_metrics['rmse'],
                'test_r2': test_metrics['r2'],
                'overfit_gap_rmse': overfit_gap_rmse,
                'overfit_ratio_rmse': overfit_ratio_rmse,
                'run_id': run.info.run_id,
            }
        )

results_df = pd.DataFrame(results).sort_values(['test_rmse', 'cv_val_rmse_mean']).reset_index(drop=True)
results_df['global_rank'] = np.arange(1, len(results_df) + 1)
results_df['family_rank'] = (
    results_df.groupby('search_family')['cv_val_rmse_mean']
    .rank(method='dense', ascending=True)
    .astype(int)
)
results_df.to_csv(MODEL_RESULTS_PATH, index=False)

family_best_df = (
    results_df.sort_values(['search_family', 'cv_val_rmse_mean', 'test_rmse'])
    .groupby('search_family', as_index=False)
    .first()
)
family_best_df.to_csv(FAMILY_RESULTS_PATH, index=False)

with mlflow.start_run(run_name='experience_1__summary'):
    mlflow.log_param('experience_name', 'experience_1')
    mlflow.log_param('models_tested', ','.join(candidate_models.keys()))
    mlflow.log_param('selected_feature_strategy', 'no_area_plus_recent_3_yield_years')
    mlflow.log_param('cross_validation_strategy', 'GroupKFold(area)_on_train_only')
    mlflow.log_metric('best_test_rmse', float(results_df.loc[0, 'test_rmse']))
    mlflow.log_metric('best_test_r2', float(results_df.loc[0, 'test_r2']))
    mlflow.log_metric('best_cv_val_rmse_mean', float(results_df.loc[0, 'cv_val_rmse_mean']))
    mlflow.log_artifact(str(MODEL_RESULTS_PATH))
    mlflow.log_artifact(str(FAMILY_RESULTS_PATH))
    mlflow.log_artifact(str(SEARCH_SPACE_PATH))

print(f'Résultats sauvegardés : {MODEL_RESULTS_PATH.resolve()}')
print(f'Meilleurs modèles par famille : {FAMILY_RESULTS_PATH.resolve()}')


2026/05/04 17:23:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/04 17:23:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/04 17:23:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Résultats sauvegardés : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_1/model_results.csv
Meilleurs modèles par famille : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_1/family_best_results.csv


## Lecture finale

Cette expérience ne se limite plus au couple `train/test` du split final.
Elle permet maintenant de lire l'overfitting à trois niveaux :

- `train` : qualité d'ajustement sur l'échantillon d'entraînement ;
- `cv_val` : stabilité sur des folds groupés hors échantillon à l'intérieur du train ;
- `test` : performance finale sur des `area` totalement absentes du train.

La recherche ciblée doit être lue avec la hiérarchie suivante :
- d'abord repérer le meilleur candidat **par famille** dans `family_best_results.csv` ;
- ensuite comparer ces meilleurs candidats entre eux sur `test_rmse` et `cv_val_rmse_mean` ;
- enfin arbitrer entre performance brute et robustesse via `overfit_ratio_rmse` et `cv_overfit_ratio_rmse_mean`.


In [8]:
display(results_df)
display(pd.read_csv(FAMILY_RESULTS_PATH))
print('Configuration retenue : area retirée des features, historique de rendement réduit à 3 années récentes.')
print(f'Années de rendement retenues : {SELECTED_YIELD_YEARS}')
print(f'Folds de cross-validation groupée : {CV_N_SPLITS}')
print(f"Meilleur modèle global : {results_df.loc[0, 'model']}")
print(f"Famille : {results_df.loc[0, 'search_family']}")
print(f"Étape de tuning : {results_df.loc[0, 'tuning_stage']}")
print(f"CV validation RMSE moyen : {results_df.loc[0, 'cv_val_rmse_mean']:.4f} +/- {results_df.loc[0, 'cv_val_rmse_std']:.4f}")
print(f"Test RMSE : {results_df.loc[0, 'test_rmse']:.4f}")
print(f"Ratio d'overfit final : {results_df.loc[0, 'overfit_ratio_rmse']:.4f}")
print(f"Run MLflow du meilleur modèle : {results_df.loc[0, 'run_id']}")


,model,model_family,search_family,tuning_stage,regularization_profile,train_mae,train_rmse,train_r2,cv_val_mae_mean,cv_val_mae_std,cv_val_rmse_mean,cv_val_rmse_std,cv_val_r2_mean,cv_overfit_gap_rmse_mean,cv_overfit_ratio_rmse_mean,test_mae,test_rmse,test_r2,overfit_gap_rmse,overfit_ratio_rmse,run_id,global_rank,family_rank
0,random_forest_search_01,random_forest,random_forest_focus,targeted_search,manual_targeted_search,0.414784,1.119459,0.981298,0.640269,0.038573,1.591890,0.248256,0.961808,0.457752,1.420163,0.803685,2.056729,0.946969,0.937270,1.837253,ac8b6a704b864909b2b6495f78063f33,1,3
1,random_forest_search_03,random_forest,random_forest_focus,targeted_search,manual_targeted_search,0.440142,1.169191,0.979599,0.654021,0.044487,1.608053,0.260967,0.961071,0.408259,1.352384,0.800951,2.058856,0.946859,0.889665,1.760923,26cb08b3abe24a7aad1f1a595898aad5,2,4
2,random_forest_search_02,random_forest,random_forest_focus,targeted_search,manual_targeted_search,0.454953,1.195743,0.978662,0.643910,0.041618,1.620877,0.252040,0.960291,0.389204,1.329654,0.794532,2.068329,0.946369,0.872586,1.729744,bb5cae0eaa2446e1b8daae5f7721aa15,3,5
3,random_forest_regularized,random_forest,random_forest_focus,seed_regularized,seed_regularized,0.408806,1.090673,0.982247,0.634725,0.037829,1.566738,0.245997,0.962891,0.472936,1.451185,0.807923,2.098300,0.944804,1.007627,1.923858,af5da396df5b4a079415f8529e4b1146,4,1
4,xgboost_regularized,xgboost,xgboost_focus,seed_regularized,seed_regularized,0.421038,0.869498,0.988717,0.725244,0.051841,1.674089,0.258937,0.957414,0.821906,1.980277,0.854425,2.147721,0.942173,1.278223,2.470069,fe1274c42c9940b99838c7ec33fd25d8,5,2
5,xgboost_random_forest,xgboost_random_forest,xgboost_random_forest_focus,baseline,baseline,0.566441,1.298934,0.974820,0.825121,0.082919,1.947996,0.298219,0.942897,0.565703,1.404827,0.891290,2.193577,0.939677,0.894643,1.688751,a230ba3140894ac481800268e1ec9782,6,1
6,random_forest,random_forest,random_forest_focus,baseline,baseline,0.293445,0.850912,0.989194,0.630194,0.055499,1.569108,0.300556,0.961885,0.729584,1.913381,0.837602,2.238686,0.937171,1.387774,2.630924,3655fe747933406588fef0ad49bf742c,7,2
7,xgboost,xgboost,xgboost_focus,baseline,baseline,0.112001,0.163362,0.999602,0.708255,0.032868,1.661645,0.245509,0.958345,1.537564,13.651412,0.925761,2.318342,0.932620,2.154980,14.191420,24d0d6906b4f46e8b90e89b457fc4eed,8,1
8,xgboost_random_forest_regularized,xgboost_random_forest,xgboost_random_forest_focus,seed_regularized,seed_regularized,0.863797,1.998291,0.940406,1.013252,0.132989,2.286807,0.429462,0.921233,0.161514,1.080070,1.059912,2.343115,0.931173,0.344824,1.172560,7580e6d4801043aaaef542e375d86139,9,2
9,xgboost_random_forest_search_01,xgboost_random_forest,xgboost_random_forest_focus,targeted_search,manual_targeted_search,0.963140,2.211741,0.926995,1.082278,0.116723,2.430035,0.417012,0.911342,0.129455,1.060834,1.144127,2.447886,0.924880,0.236145,1.106769,926dab63ed5e4019b1807f95e0857938,10,4


,search_family,model,model_family,tuning_stage,regularization_profile,train_mae,train_rmse,train_r2,cv_val_mae_mean,cv_val_mae_std,cv_val_rmse_mean,cv_val_rmse_std,cv_val_r2_mean,cv_overfit_gap_rmse_mean,cv_overfit_ratio_rmse_mean,test_mae,test_rmse,test_r2,overfit_gap_rmse,overfit_ratio_rmse,run_id,global_rank,family_rank
0,random_forest_focus,random_forest_regularized,random_forest,seed_regularized,seed_regularized,0.408806,1.090673,0.982247,0.634725,0.037829,1.566738,0.245997,0.962891,0.472936,1.451185,0.807923,2.098300,0.944804,1.007627,1.923858,af5da396df5b4a079415f8529e4b1146,4,1
1,xgboost_focus,xgboost,xgboost,baseline,baseline,0.112001,0.163362,0.999602,0.708255,0.032868,1.661645,0.245509,0.958345,1.537564,13.651412,0.925761,2.318342,0.932620,2.154980,14.191420,24d0d6906b4f46e8b90e89b457fc4eed,8,1
2,xgboost_random_forest_focus,xgboost_random_forest,xgboost_random_forest,baseline,baseline,0.566441,1.298934,0.974820,0.825121,0.082919,1.947996,0.298219,0.942897,0.565703,1.404827,0.891290,2.193577,0.939677,0.894643,1.688751,a230ba3140894ac481800268e1ec9782,6,1


Configuration retenue : area retirée des features, historique de rendement réduit à 3 années récentes.
Années de rendement retenues : [2013, 2014, 2015]
Folds de cross-validation groupée : 4
Meilleur modèle global : random_forest_search_01
Famille : random_forest_focus
Étape de tuning : targeted_search
CV validation RMSE moyen : 1.5919 +/- 0.2483
Test RMSE : 2.0567
Ratio d'overfit final : 1.8373
Run MLflow du meilleur modèle : ac8b6a704b864909b2b6495f78063f33


## Export explicite du pipeline historique `P1`

Cette section fige l'artefact reutilisable par `notebooks/experience_3.ipynb` : le pipeline historique complet qui calcule `P1`, plus un fichier de metadonnees exploitable ensuite par l'API.

In [9]:
from datetime import datetime, timezone
import joblib

MODELS_DIR = ARTIFACTS_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
P1_MODEL_PATH = MODELS_DIR / 'p1_historical_pipeline.joblib'
P1_METADATA_PATH = MODELS_DIR / 'p1_historical_metadata.json'

best_model_name = str(results_df.loc[0, 'model'])
best_model_spec = candidate_models[best_model_name]
p1_training_X = model_df[feature_cols].copy()
p1_training_y = y.copy()

p1_historical_pipeline = build_model_pipeline(best_model_spec['estimator'])
p1_historical_pipeline.fit(p1_training_X, p1_training_y)

p1_metadata = {
    'artifact_role': 'P1_historical_prediction_model',
    'training_notebook': 'notebooks/experience_1.ipynb',
    'model_name': best_model_name,
    'model_family': best_model_spec['model_family'],
    'search_family': best_model_spec['search_family'],
    'tuning_stage': best_model_spec['tuning_stage'],
    'regularization_profile': best_model_spec['regularization_profile'],
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_source': str(EXPERIENCE_DATASET_PATH),
    'target_year': int(TARGET_YEAR),
    'target_column': target_col,
    'feature_columns': feature_cols,
    'selected_yield_years': SELECTED_YIELD_YEARS,
    'area_role': 'group_only_not_feature',
    'split_strategy': 'GroupShuffleSplit(area, test_size=0.2, random_state=42)',
    'metrics': {
        'test_rmse': float(results_df.loc[0, 'test_rmse']),
        'test_mae': float(results_df.loc[0, 'test_mae']),
        'test_r2': float(results_df.loc[0, 'test_r2']),
        'cv_val_rmse_mean': float(results_df.loc[0, 'cv_val_rmse_mean']),
        'cv_val_mae_mean': float(results_df.loc[0, 'cv_val_mae_mean']),
        'cv_val_r2_mean': float(results_df.loc[0, 'cv_val_r2_mean']),
    },
    'mlflow_run_id': str(results_df.loc[0, 'run_id']) if 'run_id' in results_df.columns else None,
}

joblib.dump(p1_historical_pipeline, P1_MODEL_PATH)
P1_METADATA_PATH.write_text(json.dumps(p1_metadata, indent=2, ensure_ascii=True), encoding='utf-8')

p1_metadata_preview_df = pd.DataFrame(
    {
        'metadata_key': list(p1_metadata.keys()),
        'value': [json.dumps(value, ensure_ascii=True) if isinstance(value, (dict, list)) else value for value in p1_metadata.values()],
    }
)
display(p1_metadata_preview_df)
print(f'Pipeline historique P1 sauvegarde : {P1_MODEL_PATH.resolve()}')
print(f'Metadonnees P1 sauvegardees : {P1_METADATA_PATH.resolve()}')


,metadata_key,value
0,artifact_role,P1_historical_prediction_model
1,training_notebook,notebooks/experience_1.ipynb
2,model_name,random_forest_search_01
3,model_family,random_forest
4,search_family,random_forest_focus
5,tuning_stage,targeted_search
6,regularization_profile,manual_targeted_search
7,trained_at_utc,2026-05-04T15:24:36.220043+00:00
8,dataset_source,/Users/steph/Code/Python/Jupyter/OCR_Projet12/...
9,target_year,2016


Pipeline historique P1 sauvegarde : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/models/p1_historical_pipeline.joblib
Metadonnees P1 sauvegardees : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/models/p1_historical_metadata.json
